In [1]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT / "src"))
ART = ROOT / "artifacts"
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
from plotting import series_styles, new_figure

plt.rcParams.update({"figure.dpi": 110, "font.size": 9})

def es(v, d=4):
    """Numero con formato de Colombia: punto de miles y coma decimal."""
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return ""
    if isinstance(v, (int, np.integer)):
        return f"{int(v):,}".replace(",", ".")
    s = f"{float(v):,.{d}f}"
    return s.replace(",", "X").replace(".", ",").replace("X", ".")

def tabla(df, d=4, enteros=()):
    out = df.copy()
    for c in out.columns:
        if c in enteros:
            out[c] = out[c].map(lambda v: es(int(v)) if pd.notna(v) else "")
        elif pd.api.types.is_float_dtype(out[c]):
            out[c] = out[c].map(lambda v: es(v, d))
        elif pd.api.types.is_integer_dtype(out[c]):
            out[c] = out[c].map(es)
    display(out.style.hide(axis="index"))

def figura(ruta, ancho=None):
    display(Image(filename=str(ruta), width=ancho))

# 10. Reflexión crítica y conclusiones

Este capítulo responde, con la evidencia cuantitativa de los capítulos anteriores, las seis preguntas de reflexión crítica del enunciado, contrasta las preguntas de investigación P1 a P4 y las hipótesis H1 y H2, y cierra con las conclusiones del proyecto. Cada respuesta remite a la sección donde está la evidencia, para que pueda verificarse.

## 10.1 ¿Qué entorno fue más rápido, en qué rango de volumen y por qué?

scikit-learn fue más rápido en todos los volúmenes en que pudo correr, si se mide el flujo completo. Con la misma entrada de 800.847 filas, leer, preprocesar, entrenar y predecir con la regresión logística tomó 32,4 s en scikit-learn y 47,3 s en Spark, y la mayor diferencia estuvo en la lectura con caché, 0,85 s frente a 11,7 s (sección 9.2). Si se mide solo el ajuste del modelo, scikit-learn gana con claridad en volúmenes pequeños, 1,1 s frente a 2,6 s con 80 mil filas, y Spark lo alcanza entre unas 400 mil y 1 millón de filas, sin que el punto pueda fijarse con más precisión, porque con la misma muestra de 800 mil filas el orden se invirtió entre dos corridas (sección 7.4). Desde 1,6 millones de filas, scikit-learn no cupo en la memoria disponible de la máquina, mientras que Spark entrenó la regresión logística con los 32,4 millones de filas de train en unos 11 minutos.

La razón es estructural. scikit-learn trabaja en un solo proceso, sobre matrices dispersas en memoria y sin costo de coordinación, y su tiempo crece de forma casi lineal con las filas. Spark paga un costo fijo por cada tarea y por cada iteración de L-BFGS, unas 40 rondas de agregación por ajuste, y ese costo domina con pocos datos; con más datos se reparte y su tiempo crece de forma sublineal ($b = 0{,}54$). En esta máquina, además, Spark no convierte núcleos adicionales en velocidad: su óptimo está en `local[2]` y con 12 núcleos tarda más que con uno (sección 7.2).

## 10.2 ¿Cuál fue más preciso y calibrado, y es estadísticamente significativa la diferencia?

El entorno más preciso fue scikit-learn. Su MLP alcanzó un AUC medio de 0,7306 con 1,6 millones de filas, frente a 0,7157 de la regresión logística de Spark, el mejor modelo de ese entorno, y 0,6607 del MLP de Spark (sección 9.1). La diferencia entre los MLP de ambos entornos, evaluados sobre las mismas 4.218.938 filas del día 30, es de 0,0720, con IC 95 % de DeLong de 0,0714 a 0,0725, y se sostiene en las 5 semillas (sección 6.5). Es significativa en los dos niveles de inferencia y 7 veces mayor que el margen $\delta = 0{,}01$, así que también es relevante en la práctica.

En calibración, los tres modelos de scikit-learn y la regresión logística de Spark están bien calibrados, con ECE entre 0,0073 y 0,0089, y el MLP y el GBT de Spark no lo están (ECE de 0,0469 y 0,0289, sección 6.2). Dentro de scikit-learn, la ventaja del MLP es pequeña: frente a HistGB es de 0,0018 y es equivalente dentro de $\delta$ (TOST con *p* de Holm de 0,0004), y frente a SGD es de 0,0041, significativa (*p* de Holm de 0,0076) pero menor que $\delta$. Es decir, la diferencia entre entornos es grande y robusta, mientras que la diferencia entre el MLP y líneas base bien afinadas es real pero de poca relevancia.

## 10.3 ¿Cuándo se justifica PySpark y cuándo es sobrecosto?

**Volumen.** PySpark se justifica cuando los datos no caben en la memoria de una máquina. En este proyecto, el pico de preprocesamiento de scikit-learn fue de unos 2,6 KB por fila de train (sección 7.1), lo que en una máquina de 16 GB deja el límite práctico alrededor de 1,6 millones de filas: el experimento principal pudo entrenar con ese tamaño, pero en la medición del punto de cruce, con el navegador y el editor abiertos, ya no cabía. Spark, en cambio, procesó los 32,4 millones de filas del train completo con la regresión logística. Por debajo de ese límite, Spark es sobrecosto: tarda más en el flujo completo, consume más memoria (2,8 GB del proceso Java frente a 2,0 GB de Python con 800 mil filas) y no mejora la precisión, porque el mejor modelo de Spark quedó 1,5 puntos de AUC por debajo del de scikit-learn.

**Tolerancia a fallos.** Spark recupera particiones perdidas recalculándolas a partir de su linaje, lo que importa en trabajos largos sobre clústeres donde un nodo puede fallar. En modo local esa ventaja no existe, porque todo corre en una sola JVM: durante este proyecto, una falla de comunicación entre Python y el proceso Java (`SocketTimeoutException`) obligó a repetir la medición completa del punto de cruce. La tolerancia a fallos solo justifica Spark cuando hay un clúster real.

**Ecosistema.** Spark se justifica cuando los datos ya viven en un lago de datos o en un flujo de Spark SQL y mover el modelo a otro entorno cuesta más que entrenarlo allí, o cuando el modelo debe integrarse en procesos de producción que ya usan MLlib. Es sobrecosto cuando el modelo necesita arquitecturas que MLlib no ofrece, como embeddings, regularización o parada temprana en el MLP (sección 10.4), o herramientas de interpretabilidad como LIME y SHAP, que se usan con modelos de Python en memoria.

En síntesis, para el volumen de Avazu en una sola máquina, PySpark solo se justificó para entrenar la regresión logística con el train completo; para todo lo demás, scikit-learn fue más rápido, más preciso y más flexible.

## 10.4 ¿Qué limitaciones arquitectónicas impone el MLP nativo de Spark frente a una red con embeddings?

La limitación de mayor impacto medido es que `MultilayerPerceptronClassifier` densifica la entrada: el costo en memoria es $n \times d$ y no depende de cuántas columnas son distintas de cero (sección 5.3). Con el hashing de 2^12 posiciones del experimento principal, una copia densa de 1 millón de filas ocupa unos 32,8 GB, por lo que el MLP de Spark tuvo que bajar a 2^9 posiciones y 300.000 filas. Ese recorte, y no una diferencia de algoritmo, explica la mayor parte de los 7 puntos de AUC que lo separan del MLP de scikit-learn, cuya implementación opera sobre la matriz dispersa.

Una red con embeddings resolvería esa limitación de raíz. En lugar de una neurona de entrada por posición del hashing, cada categoría se representa con un vector denso aprendido de baja dimensión, de modo que el costo crece con el número de categorías por la dimensión del embedding, no con el número de filas por la dimensión de entrada. Además, las categorías con comportamiento parecido quedan cerca en el espacio del embedding y comparten información, algo que el hashing no hace, porque mezcla al azar las categorías que colisionan. A esto se suman las demás limitaciones del clasificador de Spark (sección 5.5): no tiene dropout ni regularización $L_2$, usa sigmoide fija en las capas ocultas, optimiza por lotes completos sin parada temprana por validación y su salida es un softmax de dos neuronas. El enunciado propone como extensión opcional una red con embeddings entrenada con TorchDistributor; no se desarrolló en este trabajo y queda como el siguiente paso natural.

## 10.5 ¿Qué aporta LIME, qué no puede garantizar y cómo se compara con SHAP?

LIME aporta una explicación local rápida, de unos 2 segundos por instancia, en forma de reglas legibles como `app_id_te > 0.42`, y reproducible si se fija el número de muestras: con 5.000 perturbaciones, el top-5 es idéntico entre semillas y no cambia al reducir a la mitad o duplicar el ancho del kernel (sección 8.4). En los falsos positivos más confiados identifica bien el mecanismo del error: la tasa histórica de clic alta de la aplicación, que el modelo extrapola a impresiones que no la siguen.

Lo que LIME no puede garantizar es la fidelidad. El sustituto lineal reproduce el 42 % de la variabilidad local del modelo en promedio y el 35 % en los falsos negativos, donde la sigmoide de salida está saturada (sección 8.3). Además, su variable más pesada en las 12 instancias, `device_id_te`, resultó un efecto de la discretización en cuartiles: el intervalo de las instancias contiene el 99,74 % de las filas, y el peso se estima con unas 13 de las 5.000 perturbaciones (sección 8.5). Este es el hallazgo más importante del capítulo de interpretabilidad: LIME fue muy estable y, a la vez, poco fiel en su variable principal, de modo que la estabilidad no basta para validar una explicación.

Frente a SHAP, LIME comparte en promedio 3,5 de sus 5 variables principales y su ranking global tiene un $\rho$ de Spearman de 0,72 con el de SHAP. SHAP cumple la propiedad de eficiencia hasta $4{,}7 \times 10^{-11}$ y su ranking coincide con el de la importancia por permutación ($\rho = 0{,}93$), por lo que es la referencia más confiable para este modelo (sección 8.6). En ambos casos, las explicaciones son condicionales al bloque hasheado, que se mantiene fijo y no se explica.

## 10.6 Amenazas a la validez

**Deriva temporal.** La partición es temporal (train días 21 a 28, validación día 29 y prueba día 30), así que las métricas de prueba ya incluyen el efecto de la deriva de un día. El CTR diario varía entre 15,22 % y 18,30 %, la validación tiene 15,66 % y la prueba 16,93 % (sección 2.3), y `app_category` tiene un PSI de 0,184, en la franja de cambio moderado (sección 2.8). Con un solo día de prueba, no se puede estimar cómo se degrada el modelo a lo largo de varias semanas, y una parte del ECE se debe a la diferencia de CTR entre train y prueba. El riesgo es sobrestimar el desempeño en producción si la deriva crece con el tiempo.

**Fuga de información.** Todas las transformaciones, incluidos el agrupamiento de categorías raras, la frecuencia, el target encoding y el escalado, se ajustan solo con los días de train; el target encoding de train es fuera de pliegue (capítulo 3). El archivo del día 30 no se ha modificado desde que se creó, el 5 de noviembre de 2026 a las 21:37, antes de cualquier entrenamiento, y su hash SHA-256 queda en el manifiesto de datos (capítulo 11) para verificarlo; ese hash se calculó al cerrar el proyecto y no al congelar la prueba, de modo que certifica el contenido actual y su fecha de modificación, no un registro previo. La amenaza residual está en la evaluación de LIME y SHAP, que usan la muestra del día 30 como distribución de referencia; esto no afecta al modelo, que ya estaba entrenado, pero sí define el vecindario de las explicaciones.

**Anonimización de variables.** Las columnas C1 y C14 a C21 están anonimizadas y los identificadores están hasheados por el proveedor, así que no se puede interpretar su significado de negocio. El experimento principal no las usó; la ablación muestra que agregarlas junto con la hora cambia el AUC en +0,0018 (IC 95 % de −0,0028 a 0,0065), una diferencia equivalente a cero dentro de $\delta$ (sección 9.3). Además, las explicaciones solo pueden decir que pesa la tasa de clic de una aplicación concreta, no qué tipo de aplicación es.

**Sesgos de muestreo.** Los tamaños efectivos de las muestras de scikit-learn son el 80,08 % de los nominales, porque `USING SAMPLE` de DuckDB se aplica antes del filtro por día; el efecto se corrigió en los análisis posteriores y se documenta en la sección 3.6. El muestreo es aleatorio sobre los días de train, por lo que no preserva la estructura por día ni por usuario, y el muestreo de reservorio con varios hilos no es determinista, lo que obligó a fijar un hilo en los scripts finales. Los modelos de Spark se entrenaron con tamaños distintos (32,4 millones, 1 millón y 300.000 filas) por restricciones de memoria, así que la comparación entre modelos de Spark confunde el algoritmo con el volumen. Por último, el dataset es de un solo proveedor y de 10 días de 2014, de modo que los resultados no se generalizan a otros mercados de anuncios sin validación adicional.

## 10.7 Preguntas de investigación e hipótesis

| Pregunta o hipótesis | Respuesta | Evidencia |
|---|---|---|
| **P1.** ¿A partir de qué volumen PySpark supera a scikit-learn en tiempo de entrenamiento sin degradar la calidad predictiva? | En el ajuste del modelo, entre unas 400 mil y 1 millón de filas; en el flujo completo, no lo superó en ningún volumen en que scikit-learn pudo correr. Spark solo se vuelve necesario cuando los datos no caben en memoria, desde unos 1,6 millones de filas en esta máquina, y con su mejor modelo pierde 1,5 puntos de AUC | secciones 7.4, 9.1 y 9.2 |
| **H1.** Existe un punto de cruce medible, y por debajo de él scikit-learn es más eficiente. | Se confirma de forma parcial: por debajo de unas 400 mil filas, scikit-learn es claramente más eficiente, pero el cruce del ajuste cae en una franja donde la variación entre corridas, de hasta 50 %, impide fijarlo con precisión | sección 7.4 |
| **P2.** ¿Cómo afecta la codificación de alta cardinalidad al desempeño y al costo? | Solo importa si borra la identidad de la categoría: la frecuencia sola pierde 0,146 de AUC. Hashing solo, target encoding solo y la combinación son equivalentes dentro de $\delta$. El costo depende de la dimensión del hashing: el tiempo por época crece casi en proporción a ella y la memoria de la matriz no cambia | secciones 9.3 y 9.4 |
| **H2.** El hashing con dimensión moderada es comparable a la frecuencia con una fracción de la memoria. | Se rechaza: el hashing supera a la frecuencia en 0,146 de AUC y usa 1,5 veces más memoria en la matriz de entrenamiento | sección 9.4 |
| **P3.** ¿Aporta el MLP una mejora estadísticamente significativa sobre líneas base más simples? | Frente a la regresión logística con SGD, sí, pero de 0,0041 de AUC, menor que $\delta$; frente a HistGB es equivalente dentro de $\delta$. En Spark, el MLP es 0,055 peor que la regresión logística | sección 6.5 |
| **P4.** ¿Son las explicaciones de LIME estables y fieles al modelo? | Estables sí: con 5.000 muestras el top-5 no cambia entre semillas ni con el ancho del kernel. Fieles solo en parte: $R^2$ medio de 0,42, una variable principal que resulta un efecto de la discretización y 3,5 de 5 variables en común con SHAP | capítulo 8 |

## 10.8 Conclusiones

El MLP de una capa oculta entrenado con scikit-learn fue el mejor modelo del proyecto, con un AUC de 0,7306 sobre el día de prueba, bien calibrado y con un lift de 2,39 en el primer decil. Su ventaja sobre líneas base bien afinadas es real pero pequeña: equivalente a HistGB dentro de $\delta = 0{,}01$ y 0,4 puntos por encima de la regresión logística. Con variables categóricas de identidad, como las de Avazu, la mayor parte de la señal está en la tasa histórica de clic de la aplicación y del sitio, y un modelo lineal bien codificado ya la captura casi toda.

La comparación entre entornos no favorece a Spark en una sola máquina. scikit-learn fue más rápido en el flujo completo, más preciso en 7 puntos de AUC frente al MLP de Spark y más flexible, porque Spark obligó a densificar la entrada del MLP, a reducir su muestra a 300 mil filas y a dejar el árbol potenciado sin las variables de alta cardinalidad. El escalamiento local mostró que Spark no aprovecha más de dos núcleos en este procesador, que la ley de Amdahl no describe una curva que empeora con más núcleos y que la carga está limitada por la coordinación, sin *spill* ni sesgo entre particiones. La ventaja concreta de Spark fue entrenar con los 32,4 millones de filas de train, algo que scikit-learn no podía hacer en 16 GB.

Las ablaciones mostraron que el modelo principal está sobredimensionado en algunos frentes sin pérdida de desempeño: el hashing de 2^10 rinde igual que el de 2^12 con una quinta parte del tiempo por época, y ni el cruce de variables ni la segunda capa aportan. La ponderación de clases dejó una lección metodológica: su primera versión perdió 0,040 de AUC, y no por ponderar, sino porque el sobremuestreo previo contaminaba la validación de la parada temprana y el criterio de accuracy no detecta el sobreajuste con clases balanceadas.

En interpretabilidad, LIME resultó estable pero poco fiel: su explicación más repetida, `device_id_te`, es un efecto de la discretización en cuartiles, estimado con unas 13 de 5.000 perturbaciones. SHAP y la importancia por permutación coinciden entre sí y explican los falsos positivos por la tasa de clic histórica de la aplicación. La conclusión práctica es que una explicación local solo es confiable si se reporta con su fidelidad y se contrasta con otro método.

Las principales limitaciones son el uso de una sola máquina de 16 GB, sin clúster; un solo día de prueba, que no permite medir la degradación del modelo en el tiempo; y la variación entre corridas de un portátil, que limita la precisión de las mediciones de tiempo. El trabajo futuro más relevante es una red con embeddings entrenada de forma distribuida, que eliminaría la densificación del MLP de Spark, y la evaluación sobre varios días de prueba consecutivos.

## Bibliografía

```{bibliography}
:style: plain
```